# UKB Showcase+ BERTopic with seed-robustness testing

This Colab notebook:

1. mounts Google Drive and optionally uploads the full-endpoint Showcase+ parquet;
2. loads and validates publication text and years;
3. creates or reuses cached SPECTER embeddings;
4. evaluates the original BERTopic parameter grid over multiple UMAP seeds;
5. measures seed stability using pairwise Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI);
6. records HDBSCAN cluster persistence as an internal structural-robustness diagnostic;
7. selects a robust parameter configuration and representative seed;
8. fits the final global BERTopic model;
9. exports the requested two-column `id, topics` file;
10. generates topic-over-time tables, a streamgraph, and ridge-wave figures; and
11. saves sampled interactive 3D UMAP views for all topics, persistence, and the selected wave topics.

Year is used only for topic-over-time summaries. It is not supplied as a feature when assigning a paper to a topic.


In [1]:
%pip -q install bertopic sentence-transformers umap-learn hdbscan gensim pyarrow plotly kaleido


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 5.6 MB/s eta 0:00:00


## 4. Analysis configuration

The default robustness run fits five parameter combinations across five seeds: 25 BERTopic fits. Embeddings are computed only once.

For a quick pipeline check, set `GRID_MAX_DOCUMENTS` to a smaller number such as `8000`. For the final robustness analysis, leave it as `None` to use all eligible papers.


In [5]:
MIN_YEAR = 2014
INCOMPLETE_YEARS = [2026]
EMBEDDING_MODEL_NAME = "allenai-specter"
EMBEDDING_BATCH_SIZE = 64

SEEDS = [11, 21, 42, 73, 101]
PARAM_GRID = [
    {"n_neighbors": 15, "min_cluster_size": 25, "min_samples": 5},
    {"n_neighbors": 25, "min_cluster_size": 35, "min_samples": 10},
    {"n_neighbors": 35, "min_cluster_size": 45, "min_samples": 10},
    {"n_neighbors": 45, "min_cluster_size": 55, "min_samples": 15},
    {"n_neighbors": 25, "min_cluster_size": 60, "min_samples": 20},
]
GRID_MAX_DOCUMENTS = None
TOP_N_TOPICS_FOR_FIGURES = 14
MIN_TOPIC_SIZE_FOR_EMERGENCE = 100

QUALITY_WEIGHTS = {
    "topic_diversity": 0.25,
    "outlier_rate": -0.80,
}
ROBUSTNESS_WEIGHTS = {
    "mean_ari": 0.25,
    "mean_nmi": 0.10,
    "topic_count_cv": -0.10,
}
PERSISTENCE_SENSITIVITY_WEIGHT = 0.15
SELECT_WITH_PERSISTENCE = False
LOW_PERSISTENCE_THRESHOLD = 0.10
VISUALISATION_MAX_POINTS = 15000


## 5. Imports and helper functions


In [6]:
import gc
import hashlib
import itertools
import json
import re
import textwrap
import warnings
from collections import defaultdict

import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from hdbscan import HDBSCAN
from IPython.display import display
from scipy.ndimage import gaussian_filter1d
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from umap import UMAP

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

CUSTOM_STOPWORDS = sorted(
    set(ENGLISH_STOP_WORDS)
    | {
        "study", "studies", "result", "results", "method", "methods",
        "conclusion", "conclusions", "background", "objective", "objectives",
        "analysis", "analyses", "data", "using", "used", "use", "based",
        "association", "associations", "associated", "effect", "effects",
        "participant", "participants", "uk", "ukb", "biobank", "united",
        "kingdom", "paper", "research", "et", "al",
    }
)

def clean_value(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return str(value).strip()

def clean_topic_text(title, abstract=""):
    text = f"{clean_value(title)}. {clean_value(abstract)}".strip()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\b©\s*\d{4}.*", " ", text)
    text = re.sub(r"\bCopyright\b.*", " ", text, flags=re.I)
    text = re.sub(
        r"\b(Background|Objective|Objectives|Aim|Aims|Methods|Results|Conclusion|Conclusions|Keywords|Funding|Conflict of interest)\s*:",
        " ",
        text,
        flags=re.I,
    )
    return re.sub(r"\s+", " ", text).strip()

def infer_column(frame, candidates, label, required=True):
    lower = {str(column).lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    if required:
        raise KeyError(f"Could not identify {label} column. Tried: {candidates}")
    return None

def extract_year(frame, year_column, date_column):
    if year_column:
        years = pd.to_numeric(frame[year_column], errors="coerce")
    else:
        years = pd.Series(np.nan, index=frame.index)
    if date_column:
        fallback = frame[date_column].astype(str).str.extract(r"((?:19|20)\d{2})", expand=False)
        years = years.fillna(pd.to_numeric(fallback, errors="coerce"))
    return years.astype("Int64")

def corpus_hash(ids, docs):
    digest = hashlib.sha1()
    for paper_id, document in zip(ids, docs):
        digest.update(str(paper_id).encode("utf-8", errors="ignore"))
        digest.update(b"\t")
        digest.update(document[:1000].encode("utf-8", errors="ignore"))
        digest.update(b"\n")
    return digest.hexdigest()[:12]

def build_vectorizer():
    return CountVectorizer(
        stop_words=CUSTOM_STOPWORDS,
        ngram_range=(1, 3),
        min_df=5,
        max_df=0.85,
    )

def build_topic_model(params, seed):
    umap_model = UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=seed,
        low_memory=True,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=build_vectorizer(),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        representation_model=MaximalMarginalRelevance(diversity=0.35),
        calculate_probabilities=False,
        verbose=False,
    )

def reduce_outliers(topic_model, docs, topics, embeddings):
    topics = list(topics)
    if -1 not in topics:
        return topics
    try:
        reassigned = topic_model.reduce_outliers(
            docs,
            topics,
            strategy="embeddings",
            embeddings=embeddings,
        )
        topic_model.update_topics(docs, topics=reassigned, vectorizer_model=build_vectorizer())
        return list(reassigned)
    except Exception as error:
        print(f"Outlier reassignment skipped: {error}")
        return topics

def topic_words(topic_model, top_n=10):
    result = {}
    for topic_id, weighted_words in topic_model.get_topics().items():
        if int(topic_id) == -1:
            continue
        words = [word for word, _ in weighted_words[:top_n] if str(word).strip()]
        if words:
            result[int(topic_id)] = words
    return result

def tokenise_documents(docs):
    return [
        [
            token
            for token in re.findall(r"[a-zA-Z][a-zA-Z\-]{2,}", document.lower())
            if token not in CUSTOM_STOPWORDS
        ]
        for document in docs
    ]

def coherence_cv(tokenised_docs, dictionary, words):
    if not words or len(dictionary) == 0:
        return np.nan
    return float(
        CoherenceModel(
            topics=list(words.values()),
            texts=tokenised_docs,
            dictionary=dictionary,
            coherence="c_v",
        ).get_coherence()
    )

def topic_diversity(words):
    flattened = [word for values in words.values() for word in values]
    return len(set(flattened)) / len(flattened) if flattened else np.nan

def cluster_persistence_summary(hdbscan_model, raw_assignments):
    persistence = np.asarray(hdbscan_model.cluster_persistence_, dtype=float)
    raw_assignments = np.asarray(raw_assignments, dtype=int)
    cluster_ids = np.arange(len(persistence), dtype=int)
    sizes = np.asarray([(raw_assignments == cluster_id).sum() for cluster_id in cluster_ids])
    table = pd.DataFrame(
        {
            "topic": cluster_ids,
            "cluster_persistence": persistence,
            "raw_cluster_size": sizes,
        }
    )
    if len(table) == 0:
        metrics = {
            "mean_cluster_persistence": np.nan,
            "median_cluster_persistence": np.nan,
            "weighted_cluster_persistence": np.nan,
            "minimum_cluster_persistence": np.nan,
            "low_persistence_fraction": np.nan,
        }
    else:
        metrics = {
            "mean_cluster_persistence": float(persistence.mean()),
            "median_cluster_persistence": float(np.median(persistence)),
            "weighted_cluster_persistence": float(np.average(persistence, weights=np.maximum(sizes, 1))),
            "minimum_cluster_persistence": float(persistence.min()),
            "low_persistence_fraction": float(np.mean(persistence < LOW_PERSISTENCE_THRESHOLD)),
        }
    return metrics, table

def topic_count_penalty(n_topics):
    if n_topics < 8:
        return (8 - n_topics) * 0.05
    if n_topics > 80:
        return (n_topics - 80) * 0.01
    return 0.0

def quality_score(n_topics, outlier_rate, coherence, diversity):
    coherence = 0.0 if pd.isna(coherence) else float(coherence)
    diversity = 0.0 if pd.isna(diversity) else float(diversity)
    return (
        coherence
        + QUALITY_WEIGHTS["topic_diversity"] * diversity
        + QUALITY_WEIGHTS["outlier_rate"] * outlier_rate
        - topic_count_penalty(n_topics)
    )

def save_figure(figure, basename):
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"{basename}.png", dpi=300, bbox_inches="tight")
    figure.savefig(OUTPUT_DIR / f"{basename}.pdf", bbox_inches="tight")
    plt.close(figure)


## 6. Load and validate the publication parquet


In [7]:
raw = pd.read_parquet(PARQUET_PATH)

id_column = infer_column(raw, ["id", "publication_id", "pid", "showcase_plus_id"], "ID")
title_column = infer_column(raw, ["title", "original_title"], "title")
abstract_column = infer_column(raw, ["abstract", "description"], "abstract", required=False)
year_column = infer_column(raw, ["year", "publication_year"], "year", required=False)
date_column = infer_column(raw, ["date", "publication_date", "date_inserted"], "date", required=False)

publications = pd.DataFrame(
    {
        "id": raw[id_column].map(clean_value),
        "title": raw[title_column].map(clean_value),
        "abstract": raw[abstract_column].map(clean_value) if abstract_column else "",
        "year": extract_year(raw, year_column, date_column),
    }
)
publications["topic_text"] = [
    clean_topic_text(title, abstract)
    for title, abstract in zip(publications["title"], publications["abstract"])
]
publications = publications[
    publications["id"].ne("")
    & publications["topic_text"].str.len().gt(20)
    & publications["year"].ge(MIN_YEAR)
].copy()
publications = publications.drop_duplicates("id", keep="first").reset_index(drop=True)

assert publications["id"].is_unique
assert publications["topic_text"].str.len().gt(20).all()
assert publications["year"].notna().all()

print(f"Input rows: {len(raw):,}")
print(f"Eligible unique publications: {len(publications):,}")
print(f"Years: {publications['year'].min()}–{publications['year'].max()}")
display(publications[["id", "title", "year"]].head())


Input rows: 26,109
Eligible unique publications: 26,103
Years: 2014–2026


,id,title,year
0,pub.1000169743,Ethnic-Specific Obesity Cutoffs for Diabetes R...,2014
1,pub.1000305094,Data science for mental health: a UK perspecti...,2016
2,pub.1001609820,"Ambient air pollution, traffic noise and adult...",2016
3,pub.1001822430,Correction: Cognitive Test Scores in UK Bioban...,2016
4,pub.1001987017,"0154 A new, efficient web-based tool to collec...",2014


## 7. Create or reuse SPECTER embeddings

The cache key depends on the paper IDs and text. Changing the corpus creates a new cache file automatically.


In [8]:
docs = publications["topic_text"].tolist()
paper_ids = publications["id"].tolist()
cache_key = corpus_hash(paper_ids, docs)
embedding_path = CACHE_DIR / f"embeddings_allenai-specter_{len(docs)}docs_{cache_key}.npy"

if embedding_path.exists():
    embeddings = np.load(embedding_path)
    print(f"Loaded cached embeddings: {embedding_path}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Encoding on {device}")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
    embeddings = embedding_model.encode(
        docs,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embedding_path, embeddings)
    print(f"Saved embeddings: {embedding_path}")

assert embeddings.shape[0] == len(publications)
print("Embedding shape:", embeddings.shape)


Loaded cached embeddings: /content/drive/MyDrive/UKB_BERTopic_seed_robustness/cache/embeddings_allenai-specter_26103docs_bf373f0d64e5.npy
Embedding shape: (26103, 768)


## 8. Prepare the robustness grid corpus

All documents are used by default. If `GRID_MAX_DOCUMENTS` is set, the same deterministic sample is used for every parameter and seed combination.


In [9]:
if GRID_MAX_DOCUMENTS and len(publications) > GRID_MAX_DOCUMENTS:
    sampling_rng = np.random.default_rng(42)
    grid_indices = np.sort(
        sampling_rng.choice(len(publications), size=GRID_MAX_DOCUMENTS, replace=False)
    )
else:
    grid_indices = np.arange(len(publications))

grid_docs = [docs[index] for index in grid_indices]
grid_embeddings = embeddings[grid_indices]
grid_tokens = tokenise_documents(grid_docs)
grid_dictionary = Dictionary(grid_tokens)

print(f"Grid documents: {len(grid_docs):,}")
print(f"Grid fits: {len(PARAM_GRID) * len(SEEDS)}")


Grid documents: 26,103
Grid fits: 25


## 9. Fit every parameter–seed combination


Rerunning the cell skips completed combinations.


In [10]:
RUNS_PATH = OUTPUT_DIR / "seed_grid_runs.csv"
ASSIGNMENT_DIR = CACHE_DIR / f"seed_assignments_{len(grid_docs)}docs_{cache_key}"
ASSIGNMENT_DIR.mkdir(parents=True, exist_ok=True)

if RUNS_PATH.exists():
    existing_runs = pd.read_csv(RUNS_PATH)
else:
    existing_runs = pd.DataFrame()

run_rows = existing_runs.to_dict("records")
completed = {
    (int(row["parameter_index"]), int(row["seed"]))
    for row in run_rows
    if str(row.get("status", "")) == "ok"
    and pd.notna(row.get("weighted_cluster_persistence"))
}

for parameter_index, params in enumerate(PARAM_GRID):
    for seed in SEEDS:
        assignment_path = ASSIGNMENT_DIR / f"params_{parameter_index}_seed_{seed}.npy"
        if (parameter_index, seed) in completed and assignment_path.exists():
            print(f"Skipping completed params={parameter_index}, seed={seed}")
            continue

        print(f"Fitting params={parameter_index}, seed={seed}: {params}")
        row = {
            "parameter_index": parameter_index,
            "seed": seed,
            **params,
            "status": "failed",
        }
        try:
            model = build_topic_model(params, seed)
            raw_assignments, _ = model.fit_transform(grid_docs, embeddings=grid_embeddings)
            persistence_metrics, _ = cluster_persistence_summary(model.hdbscan_model, raw_assignments)
            assignments = np.asarray(
                reduce_outliers(model, grid_docs, raw_assignments, grid_embeddings),
                dtype=np.int32,
            )
            np.save(assignment_path, assignments)

            ids = sorted(set(assignments) - {-1})
            n_topics = len(ids)
            outlier_rate = float(np.mean(assignments == -1))
            words = topic_words(model, top_n=10)
            diversity = topic_diversity(words)
            coherence = coherence_cv(grid_tokens, grid_dictionary, words)
            score = quality_score(n_topics, outlier_rate, coherence, diversity)

            row.update(
                {
                    "status": "ok",
                    "n_topics": n_topics,
                    "outlier_rate": outlier_rate,
                    "coherence_cv": coherence,
                    "topic_diversity": diversity,
                    "topic_count_penalty": topic_count_penalty(n_topics),
                    "quality_score": score,
                    "raw_outlier_rate": float(np.mean(np.asarray(raw_assignments) == -1)),
                    **persistence_metrics,
                }
            )
            print(
                f"topics={n_topics}, coherence={coherence:.3f}, "
                f"diversity={diversity:.3f}, outliers={outlier_rate:.3f}, "
                f"persistence={persistence_metrics['weighted_cluster_persistence']:.3f}, "
                f"quality={score:.3f}"
            )
        except Exception as error:
            row["error"] = repr(error)
            print("FAILED:", error)
        finally:
            run_rows = [
                value
                for value in run_rows
                if not (
                    int(value["parameter_index"]) == parameter_index
                    and int(value["seed"]) == seed
                )
            ]
            run_rows.append(row)
            pd.DataFrame(run_rows).sort_values(
                ["parameter_index", "seed"]
            ).to_csv(RUNS_PATH, index=False)
            if "model" in locals():
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

seed_runs = pd.read_csv(RUNS_PATH)
display(seed_runs)


Skipping completed params=0, seed=11
Skipping completed params=0, seed=21
Skipping completed params=0, seed=42
Skipping completed params=0, seed=73
Skipping completed params=0, seed=101
Skipping completed params=1, seed=11
Skipping completed params=1, seed=21
Skipping completed params=1, seed=42
Skipping completed params=1, seed=73
Skipping completed params=1, seed=101
Skipping completed params=2, seed=11
Skipping completed params=2, seed=21
Skipping completed params=2, seed=42
Fitting params=2, seed=73: {'n_neighbors': 35, 'min_cluster_size': 45, 'min_samples': 10}


2026-08-20 01:42:06,730 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


topics=104, coherence=0.778, diversity=0.766, outliers=0.000, persistence=0.098, quality=0.730
Fitting params=2, seed=101: {'n_neighbors': 35, 'min_cluster_size': 45, 'min_samples': 10}


2026-08-20 01:43:48,457 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


topics=108, coherence=0.761, diversity=0.741, outliers=0.000, persistence=0.089, quality=0.666
Fitting params=3, seed=11: {'n_neighbors': 45, 'min_cluster_size': 55, 'min_samples': 15}


...

2026-08-20 01:58:40,224 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


topics=79, coherence=0.773, diversity=0.833, outliers=0.000, persistence=0.176, quality=0.981
Fitting params=4, seed=101: {'n_neighbors': 25, 'min_cluster_size': 60, 'min_samples': 20}


2026-08-20 02:00:10,049 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


topics=71, coherence=0.764, diversity=0.849, outliers=0.000, persistence=0.190, quality=0.976


,parameter_index,seed,n_neighbors,min_cluster_size,min_samples,status,n_topics,outlier_rate,coherence_cv,topic_diversity,topic_count_penalty,quality_score,raw_outlier_rate,mean_cluster_persistence,median_cluster_persistence,weighted_cluster_persistence,minimum_cluster_persistence,low_persistence_fraction
0,0,11,15,25,5,ok,226,0.0,0.754860,0.657522,1.46,-0.540759,0.286442,0.030967,0.017039,0.056327,0.000000,0.946903
1,0,21,15,25,5,ok,225,0.0,0.753279,0.658667,1.45,-0.532054,0.283262,0.038931,0.021279,0.082858,0.000000,0.942222
2,0,42,15,25,5,ok,220,0.0,0.759037,0.663636,1.40,-0.475054,0.283646,0.032996,0.017201,0.071401,0.000000,0.945455
3,0,73,15,25,5,ok,212,0.0,0.742230,0.658962,1.32,-0.413029,0.247788,0.042049,0.021266,0.153356,0.000000,0.919811
4,0,101,15,25,5,ok,222,0.0,0.754625,0.669820,1.42,-0.497920,0.260353,0.035202,0.019807,0.075764,0.000000,0.945946
5,1,11,25,35,10,ok,118,0.0,0.776987,0.761864,0.38,0.587453,0.245489,0.058723,0.038771,0.111144,0.000000,0.889831
6,1,21,25,35,10,ok,146,0.0,0.769952,0.725342,0.66,0.291288,0.306785,0.064809,0.043006,0.113616,0.000000,0.842466
7,1,42,25,35,10,ok,139,0.0,0.784180,0.731655,0.59,0.377093,0.307359,0.064402,0.043351,0.096228,0.000000,0.841727
8,1,73,25,35,10,ok,143,0.0,0.781340,0.732867,0.63,0.334556,0.302264,0.067134,0.043740,0.111712,0.000000,0.846154
9,1,101,25,35,10,ok,136,0.0,0.768451,0.718382,0.56,0.388046,0.279163,0.065428,0.044197,0.097371,0.000000,0.823529


## 10. Calculate stability and select a robust configuration

ARI and NMI compare complete paper assignments and are unaffected by topic-label permutations. Topic-count CV measures variation in the number of discovered topics. HDBSCAN persistence measures how long each cluster survives in the density hierarchy within a fitted model; both equal-cluster and paper-count-weighted summaries are retained.

Two scores are exported: the base robust score and a persistence sensitivity score. By default, model selection uses the base score and treats persistence as a diagnostic. Set `SELECT_WITH_PERSISTENCE = True` to select using the sensitivity score, which adds the configured persistence weight. This makes the effect of persistence explicit rather than silently imposing an arbitrary weight.


In [11]:
successful_runs = seed_runs[seed_runs["status"].eq("ok")].copy()
if successful_runs.empty:
    raise RuntimeError("No successful seed-grid runs.")

stability_rows = []
seed_pair_rows = []
seed_medoid_scores = {}

for parameter_index, group in successful_runs.groupby("parameter_index"):
    assignments_by_seed = {}
    for seed in group["seed"].astype(int):
        path = ASSIGNMENT_DIR / f"params_{parameter_index}_seed_{seed}.npy"
        assignments_by_seed[seed] = np.load(path)

    pair_ari = []
    pair_nmi = []
    per_seed_ari = defaultdict(list)

    for seed_a, seed_b in itertools.combinations(sorted(assignments_by_seed), 2):
        labels_a = assignments_by_seed[seed_a]
        labels_b = assignments_by_seed[seed_b]
        ari = adjusted_rand_score(labels_a, labels_b)
        nmi = normalized_mutual_info_score(labels_a, labels_b)
        pair_ari.append(ari)
        pair_nmi.append(nmi)
        per_seed_ari[seed_a].append(ari)
        per_seed_ari[seed_b].append(ari)
        seed_pair_rows.append(
            {
                "parameter_index": parameter_index,
                "seed_a": seed_a,
                "seed_b": seed_b,
                "ari": ari,
                "nmi": nmi,
            }
        )

    mean_seed_ari = {
        seed: float(np.mean(values))
        for seed, values in per_seed_ari.items()
    }
    representative_seed = max(mean_seed_ari, key=mean_seed_ari.get)
    seed_medoid_scores[int(parameter_index)] = mean_seed_ari

    mean_quality = float(group["quality_score"].mean())
    mean_ari = float(np.mean(pair_ari))
    mean_nmi = float(np.mean(pair_nmi))
    mean_topics = float(group["n_topics"].mean())
    topic_count_cv = float(group["n_topics"].std(ddof=0) / max(mean_topics, 1))
    mean_weighted_persistence = float(group["weighted_cluster_persistence"].mean())

    robust_score = (
        mean_quality
        + ROBUSTNESS_WEIGHTS["mean_ari"] * mean_ari
        + ROBUSTNESS_WEIGHTS["mean_nmi"] * mean_nmi
        + ROBUSTNESS_WEIGHTS["topic_count_cv"] * topic_count_cv
    )
    robust_score_with_persistence = (
        robust_score + PERSISTENCE_SENSITIVITY_WEIGHT * mean_weighted_persistence
    )

    stability_rows.append(
        {
            "parameter_index": int(parameter_index),
            "n_seeds": len(group),
            "mean_quality_score": mean_quality,
            "mean_coherence_cv": float(group["coherence_cv"].mean()),
            "mean_topic_diversity": float(group["topic_diversity"].mean()),
            "mean_outlier_rate": float(group["outlier_rate"].mean()),
            "mean_raw_outlier_rate": float(group["raw_outlier_rate"].mean()),
            "mean_cluster_persistence": float(group["mean_cluster_persistence"].mean()),
            "median_cluster_persistence": float(group["median_cluster_persistence"].mean()),
            "mean_weighted_cluster_persistence": mean_weighted_persistence,
            "mean_low_persistence_fraction": float(group["low_persistence_fraction"].mean()),
            "mean_n_topics": mean_topics,
            "sd_n_topics": float(group["n_topics"].std(ddof=0)),
            "topic_count_cv": topic_count_cv,
            "mean_pairwise_ari": mean_ari,
            "min_pairwise_ari": float(np.min(pair_ari)),
            "mean_pairwise_nmi": mean_nmi,
            "min_pairwise_nmi": float(np.min(pair_nmi)),
            "representative_seed": representative_seed,
            "representative_seed_mean_ari": mean_seed_ari[representative_seed],
            "robust_score": robust_score,
            "robust_score_with_persistence": robust_score_with_persistence,
        }
    )

stability = pd.DataFrame(stability_rows)
selection_score_column = (
    "robust_score_with_persistence" if SELECT_WITH_PERSISTENCE else "robust_score"
)
stability = stability.sort_values(selection_score_column, ascending=False)
seed_pairs = pd.DataFrame(seed_pair_rows)
stability.to_csv(OUTPUT_DIR / "bertopic_seed_robustness_summary.csv", index=False)
seed_pairs.to_csv(OUTPUT_DIR / "bertopic_seed_pair_stability.csv", index=False)

best = stability.iloc[0]
BEST_PARAMETER_INDEX = int(best["parameter_index"])
BEST_PARAMS = PARAM_GRID[BEST_PARAMETER_INDEX]
BEST_SEED = int(best["representative_seed"])

print("Selected parameters:", BEST_PARAMS)
print("Representative seed:", BEST_SEED)
display(stability)


Selected parameters: {'n_neighbors': 45, 'min_cluster_size': 55, 'min_samples': 15}
Representative seed: 21


,parameter_index,n_seeds,mean_quality_score,mean_coherence_cv,mean_topic_diversity,mean_outlier_rate,mean_raw_outlier_rate,mean_cluster_persistence,median_cluster_persistence,mean_weighted_cluster_persistence,...,sd_n_topics,topic_count_cv,mean_pairwise_ari,min_pairwise_ari,mean_pairwise_nmi,min_pairwise_nmi,representative_seed,representative_seed_mean_ari,robust_score,robust_score_with_persistence
3,3,5,0.989425,0.777299,0.848503,0.0,0.223629,0.091372,0.067179,0.153349,...,2.097618,0.027968,0.907941,0.857306,0.919597,0.901949,21,0.926405,1.305573,1.328575
4,4,5,0.974400,0.765069,0.837325,0.0,0.207371,0.107177,0.079126,0.177105,...,3.187475,0.042613,0.906935,0.870199,0.923099,0.905788,11,0.917430,1.289182,1.315748
2,2,5,0.737383,0.773864,0.766077,0.0,0.270383,0.061924,0.039444,0.100565,...,7.249828,0.070524,0.647234,0.493700,0.865249,0.834482,101,0.709778,0.978664,0.993748
1,1,5,0.395687,0.776182,0.734022,0.0,0.288212,0.064099,0.042613,0.106014,...,9.810199,0.071922,0.624022,0.387383,0.879985,0.844615,42,0.682522,0.632499,0.648402
0,0,5,-0.491763,0.752806,0.661721,0.0,0.272298,0.036029,0.019318,0.087941,...,4.979960,0.022534,0.622233,0.443969,0.873307,0.843215,101,0.663937,-0.251128,-0.237937


## 11. Visualise seed robustness


In [12]:
figure, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

sns.boxplot(
    data=successful_runs,
    x="parameter_index",
    y="n_topics",
    ax=axes[0],
    color="#5B8DB8",
)
axes[0].set(title="Topic-count variation across seeds", xlabel="Parameter set", ylabel="Topics")

sns.barplot(
    data=stability,
    x="parameter_index",
    y="mean_pairwise_ari",
    ax=axes[1],
    color="#2A9D8F",
)
axes[1].set(title="Mean assignment stability", xlabel="Parameter set", ylabel="Pairwise ARI")
axes[1].set_ylim(0, 1)

sns.boxplot(
    data=successful_runs,
    x="parameter_index",
    y="weighted_cluster_persistence",
    ax=axes[2],
    color="#8E6C9E",
)
axes[2].set(
    title="HDBSCAN persistence across seeds",
    xlabel="Parameter set",
    ylabel="Paper-weighted persistence",
)
axes[2].set_ylim(0, 1)

sns.barplot(
    data=stability,
    x="parameter_index",
    y=selection_score_column,
    ax=axes[3],
    color="#FFBB00",
)
axes[3].set(title=f"Selection score: {selection_score_column}", xlabel="Parameter set", ylabel="Score")

save_figure(figure, "bertopic_seed_robustness")


## 12. Fit the final global model

The final fit uses all eligible papers, the selected parameter configuration, and the representative seed. It does not use publication year as a feature.


In [13]:
final_model = build_topic_model(BEST_PARAMS, BEST_SEED)
raw_final_topics, _ = final_model.fit_transform(docs, embeddings=embeddings)
final_persistence_metrics, final_cluster_persistence = cluster_persistence_summary(
    final_model.hdbscan_model, raw_final_topics
)
final_topics = np.asarray(
    reduce_outliers(final_model, docs, raw_final_topics, embeddings),
    dtype=np.int32,
)

final_topic_ids = sorted(set(final_topics) - {-1})
final_words = topic_words(final_model, top_n=10)
label_map = {
    topic_id: " / ".join(final_words.get(topic_id, [])[:4])
    for topic_id in final_topic_ids
}

assignments = publications[["id", "year", "title"]].copy()
assignments["topic"] = final_topics
assignments["topics"] = assignments["topic"].map(
    lambda topic_id: (
        "Outlier"
        if int(topic_id) == -1
        else f"T{int(topic_id)}: {label_map.get(int(topic_id), '')}"
    )
)

assignments.to_csv(OUTPUT_DIR / "bertopic_document_topic_assignments.csv", index=False)
assignments[["id", "topics"]].to_csv(
    OUTPUT_DIR / "showcase_plus_id_topics.csv",
    index=False,
)

topic_labels = pd.DataFrame(
    {
        "topic": final_topic_ids,
        "topic_words": [", ".join(final_words.get(topic_id, [])) for topic_id in final_topic_ids],
        "short_label": [label_map.get(topic_id, "") for topic_id in final_topic_ids],
    }
)
topic_labels.to_csv(OUTPUT_DIR / "bertopic_topic_labels.csv", index=False)
final_cluster_persistence["short_label"] = final_cluster_persistence["topic"].map(label_map)
final_cluster_persistence.to_csv(
    OUTPUT_DIR / "bertopic_final_cluster_persistence.csv", index=False
)

final_metrics = {
    "n_documents": len(assignments),
    "n_topics": len(final_topic_ids),
    "outlier_rate": float(np.mean(final_topics == -1)),
    "parameter_index": BEST_PARAMETER_INDEX,
    "parameters": BEST_PARAMS,
    "seed": BEST_SEED,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "hdbscan_persistence": final_persistence_metrics,
}
(OUTPUT_DIR / "bertopic_final_run_manifest.json").write_text(
    json.dumps(final_metrics, indent=2)
)

print(json.dumps(final_metrics, indent=2))
display(assignments[["id", "topics"]].head())


2026-08-20 02:02:08,769 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


{
  "n_documents": 26103,
  "n_topics": 76,
  "outlier_rate": 0.0,
  "parameter_index": 3,
  "parameters": {
    "n_neighbors": 45,
    "min_cluster_size": 55,
    "min_samples": 15
  },
  "seed": 21,
  "embedding_model": "allenai-specter",
  "hdbscan_persistence": {
    "mean_cluster_persistence": 0.11415317304763621,
    "median_cluster_persistence": 0.08531190834504804,
    "weighted_cluster_persistence": 0.1926080186983224,
    "minimum_cluster_persistence": 0.0014040769903171423,
    "low_persistence_fraction": 0.5921052631578947
  }
}


,id,topics
0,pub.1000169743,T26: vat / visceral / adipose / adipose tissue
1,pub.1000305094,T23: ehr / drug / knowledge / graph
2,pub.1001609820,T9: air / pollution / air pollution / pm
3,pub.1001822430,T0: dementia / cognitive / ad / alzheimers
4,pub.1001987017,T50: shift / shift work / night / workers


## 13. Final HDBSCAN persistence diagnostics

Persistence is reported per cluster and is not altered by outlier reassignment. The size-versus-persistence plot helps identify large but structurally weak topics as well as small, highly persistent topics.


In [14]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(
    data=final_cluster_persistence,
    x="cluster_persistence",
    bins=15,
    ax=axes[0],
    color="#8E6C9E",
)
axes[0].axvline(LOW_PERSISTENCE_THRESHOLD, color="#E76F51", linestyle="--")
axes[0].set(title="Final cluster persistence distribution", xlabel="HDBSCAN persistence")

sns.scatterplot(
    data=final_cluster_persistence,
    x="raw_cluster_size",
    y="cluster_persistence",
    size="raw_cluster_size",
    sizes=(30, 350),
    color="#344874",
    legend=False,
    ax=axes[1],
)
axes[1].axhline(LOW_PERSISTENCE_THRESHOLD, color="#E76F51", linestyle="--")
axes[1].set(
    title="Cluster size and structural persistence",
    xlabel="Papers in original HDBSCAN cluster",
    ylabel="HDBSCAN persistence",
)
save_figure(figure, "bertopic_final_cluster_persistence")
display(final_cluster_persistence.sort_values("cluster_persistence").head(10))


,topic,cluster_persistence,raw_cluster_size,short_label
70,70,0.001404,67,gut / ibd / uc / microbiota
60,60,0.001655,99,cad / clinical risk / chd / disease cad
23,23,0.005227,210,ehr / drug / knowledge / graph
46,46,0.009442,120,adipose / insulin / fat distribution / whr
40,40,0.012456,131,proteins / plasma proteins / proteomic / prote...
49,49,0.014422,114,segments / haplotype / ibd / haplotypes
50,50,0.015371,111,shift / shift work / night / workers
67,67,0.015708,75,mri / segmentation / body composition / automated
43,43,0.022903,121,coffee / coffee consumption / tea / coffee intake
52,52,0.024477,109,metabolites / metabolomic / metabolomics / met...


## 14. Native BERTopic topics-over-time output


In [15]:
topics_over_time = final_model.topics_over_time(
    docs,
    publications["year"].astype(int).tolist(),
    global_tuning=True,
    evolution_tuning=True,
)
topics_over_time.to_csv(
    OUTPUT_DIR / "bertopic_topics_over_time_native.csv",
    index=False,
)

native_figure = final_model.visualize_topics_over_time(topics_over_time)
native_figure.write_html(OUTPUT_DIR / "bertopic_native_topics_over_time.html")
display(topics_over_time.head())


,Topic,Words,Frequency,Timestamp,Name
0,1,"mood disorder, mood, increased rates, bipolar,...",4,2014,1_mdd_anxiety_depressive_mental
1,3,"vision, impairment, visual, eye, visual impair...",2,2014,3_retinal_glaucoma_amd_myopia
2,4,"persons, cognitive functioning, old, functioni...",1,2014,4_asthma_copd_lung function_pulmonary
3,5,"lung adenocarcinoma, adenocarcinoma, clinical ...",1,2014,5_prostate_breast_prostate cancer_breast...
4,6,"temperature, calibration, fat percentage, post...",2,2014,6_pa_mvpa_sedentary_intensity


## 15. Select 14 topics for readable wave figures

This reproduces the original selection strategy:

- take the eight largest topics by total paper count;
- rank topics with at least 100 papers by their change in annual corpus share;
- append the strongest emerging topics;
- remove duplicates; and
- stop once 14 topics have been selected.

The full paper-level output always retains every topic.


In [16]:
valid_assignments = assignments[assignments["topic"].ne(-1)].copy()

counts = (
    valid_assignments.groupby(["year", "topic"])
    .size()
    .reset_index(name="n_papers")
)
year_totals = (
    valid_assignments.groupby("year")
    .size()
    .reset_index(name="year_total")
)
counts = counts.merge(year_totals, on="year", how="left")
counts["share"] = counts["n_papers"] / counts["year_total"]

topic_totals = (
    valid_assignments["topic"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="total_n")
)

growth_rows = []
for topic_id, group in counts.groupby("topic"):
    complete = group[~group["year"].isin(INCOMPLETE_YEARS)].sort_values("year")
    if complete.empty:
        complete = group.sort_values("year")
    first = complete.iloc[0]
    last = complete.iloc[-1]
    slope = (
        np.polyfit(complete["year"], complete["share"], deg=1)[0]
        if len(complete) >= 2
        else 0.0
    )
    growth_rows.append(
        {
            "topic": int(topic_id),
            "first_year": int(first["year"]),
            "last_year": int(last["year"]),
            "first_share": float(first["share"]),
            "last_share": float(last["share"]),
            "growth_ratio": float((last["share"] + 1e-6) / (first["share"] + 1e-6)),
            "share_slope": float(slope),
        }
    )

growth = topic_totals.merge(pd.DataFrame(growth_rows), on="topic", how="left")
growth["short_label"] = growth["topic"].map(label_map)
growth = growth.sort_values(["total_n", "share_slope"], ascending=[False, False])
growth.to_csv(OUTPUT_DIR / "bertopic_topic_growth_table.csv", index=False)

largest = (
    growth.sort_values("total_n", ascending=False)
    .head(max(8, TOP_N_TOPICS_FOR_FIGURES // 2))["topic"]
    .astype(int)
    .tolist()
)
emerging = (
    growth[growth["total_n"].ge(MIN_TOPIC_SIZE_FOR_EMERGENCE)]
    .sort_values("share_slope", ascending=False)
    .head(TOP_N_TOPICS_FOR_FIGURES)["topic"]
    .astype(int)
    .tolist()
)

selected_topics = []
for topic_id in largest + emerging:
    if topic_id not in selected_topics:
        selected_topics.append(topic_id)
    if len(selected_topics) == TOP_N_TOPICS_FOR_FIGURES:
        break

selection = growth[growth["topic"].isin(selected_topics)].copy()
selection["selection_order"] = selection["topic"].map(
    {topic_id: index + 1 for index, topic_id in enumerate(selected_topics)}
)
selection["selection_source"] = selection["topic"].map(
    lambda topic_id: (
        "largest and emerging"
        if topic_id in largest and topic_id in emerging
        else "largest"
        if topic_id in largest
        else "emerging"
    )
)
selection = selection.sort_values("selection_order")
selection.to_csv(OUTPUT_DIR / "bertopic_wave_topic_selection.csv", index=False)

selected_papers = int(valid_assignments["topic"].isin(selected_topics).sum())
coverage = pd.DataFrame(
    [
        {
            "n_selected_topics": len(selected_topics),
            "selected_papers": selected_papers,
            "all_papers": len(valid_assignments),
            "coverage_percent": selected_papers / len(valid_assignments) * 100,
        }
    ]
)
coverage.to_csv(OUTPUT_DIR / "bertopic_wave_topic_coverage.csv", index=False)

print("Selected topic IDs:", selected_topics)
display(coverage)
display(selection[["selection_order", "topic", "short_label", "total_n", "selection_source"]])


Selected topic IDs: [0, 1, 2, 8, 3, 7, 6, 4, 13, 22, 9, 45, 41, 24]


,n_selected_topics,selected_papers,all_papers,coverage_percent
0,14,11473,26103,43.952802


,selection_order,topic,short_label,total_n,selection_source
0,1,0,dementia / cognitive / ad / alzheimers,3556,largest and emerging
1,2,1,mdd / anxiety / depressive / mental,1623,largest
2,3,2,sleep / sleep duration / insomnia / daytime,835,largest
3,4,8,complex traits / trait / heritability / simula...,818,largest
4,5,3,retinal / glaucoma / amd / myopia,781,largest
5,6,7,dietary / meat / food / plantbased,659,largest and emerging
6,7,6,pa / mvpa / sedentary / intensity,634,largest
7,8,4,asthma / copd / lung function / pulmonary,623,largest
14,9,13,proteins / proteomic / proteomics / plasma pro...,458,emerging
23,10,22,lpa / ascvd / lipoproteina / tyg,341,emerging


## 16. Generate streamgraph and ridge-wave figures


In [17]:
selected_counts = counts[counts["topic"].isin(selected_topics)].copy()
share_matrix = (
    selected_counts.pivot_table(
        index="year",
        columns="topic",
        values="share",
        aggfunc="sum",
        fill_value=0,
    )
    .sort_index()
    .reindex(columns=selected_topics)
)
count_matrix = (
    selected_counts.pivot_table(
        index="year",
        columns="topic",
        values="n_papers",
        aggfunc="sum",
        fill_value=0,
    )
    .sort_index()
    .reindex(columns=selected_topics)
)

display_labels = {
    topic_id: f"T{topic_id}: {label_map.get(topic_id, '')}"
    for topic_id in selected_topics
}
share_matrix_named = share_matrix.rename(columns=display_labels)
count_matrix_named = count_matrix.rename(columns=display_labels)
share_matrix_named.to_csv(OUTPUT_DIR / "bertopic_topic_year_proportions_selected.csv")
count_matrix_named.to_csv(OUTPUT_DIR / "bertopic_topic_year_counts_selected.csv")

years = share_matrix_named.index.to_numpy()
values = share_matrix_named.to_numpy().T
labels = list(share_matrix_named.columns)

figure, axis = plt.subplots(figsize=(13, 8))
axis.stackplot(years, values, labels=labels, alpha=0.88)
axis.set(
    title="Dynamic topic shares in UKB Showcase+ publications",
    xlabel="Publication year",
    ylabel="Share of all Showcase+ papers",
)
axis.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
save_figure(figure, "bertopic_dynamic_streamgraph")

if len(share_matrix_named.index) >= 3:
    smooth_values = np.apply_along_axis(
        lambda values: gaussian_filter1d(values, sigma=1.0),
        axis=0,
        arr=share_matrix_named.to_numpy(),
    )
    smooth = pd.DataFrame(
        smooth_values,
        index=share_matrix_named.index,
        columns=share_matrix_named.columns,
    )
else:
    smooth = share_matrix_named.copy()

maximum = max(float(smooth.max().max()), 1e-6)
offset = maximum * 1.25

figure, axis = plt.subplots(figsize=(13, 9))
for index, column in enumerate(smooth.columns):
    baseline = index * offset
    values = smooth[column].to_numpy()
    axis.fill_between(smooth.index, baseline, baseline + values, alpha=0.75)
    axis.plot(smooth.index, baseline + values, linewidth=1)
    axis.text(
        smooth.index.min() - 0.2,
        baseline + maximum * 0.15,
        column,
        va="center",
        ha="right",
        fontsize=8,
    )

axis.set(title="Topic evolution as ridge waves", xlabel="Publication year")
axis.set_yticks([])
axis.set_xlim(smooth.index.min() - 1, smooth.index.max() + 0.5)
save_figure(figure, "bertopic_topic_ridge_waves")


## 17. Three-dimensional UMAP and HDBSCAN diagnostics

HDBSCAN was fitted in a five-dimensional UMAP space. The coordinates below are a separate three-dimensional projection for visual inspection only; they do not replace the 5D clustering space. Because 70+ categorical colours are difficult to interpret, three complementary views are saved:

- all topic assignments, with the legend hidden but full hover information;
- cluster persistence on a continuous colour scale; and
- the 14 wave topics versus all other topics.



In [18]:
projection_path = CACHE_DIR / (
    f"umap3d_{len(embeddings)}docs_{cache_key}_params{BEST_PARAMETER_INDEX}_seed{BEST_SEED}.npy"
)
if projection_path.exists():
    coordinates_3d = np.load(projection_path)
else:
    visual_umap = UMAP(
        n_neighbors=BEST_PARAMS["n_neighbors"],
        n_components=3,
        min_dist=0.05,
        metric="cosine",
        random_state=BEST_SEED,
        low_memory=True,
    )
    coordinates_3d = visual_umap.fit_transform(embeddings)
    np.save(projection_path, coordinates_3d)

projection = assignments.copy()
projection[["umap_x", "umap_y", "umap_z"]] = coordinates_3d
persistence_map = final_cluster_persistence.set_index("topic")["cluster_persistence"].to_dict()
projection["cluster_persistence"] = projection["topic"].map(persistence_map).fillna(0.0)
projection["topic_code"] = projection["topic"].map(lambda value: f"T{int(value)}")
projection["wave_group"] = projection.apply(
    lambda row: row["topics"] if int(row["topic"]) in selected_topics else "Other topics",
    axis=1,
)

per_topic_limit = max(1, VISUALISATION_MAX_POINTS // projection["topic"].nunique())
visual_sample = pd.concat(
    [
        group.sample(n=min(len(group), per_topic_limit), random_state=BEST_SEED)
        for _, group in projection.groupby("topic")
    ],
    ignore_index=True,
)
visual_sample.to_csv(OUTPUT_DIR / "bertopic_umap3d_visualisation_sample.csv", index=False)
print(f"3D rendering sample: {len(visual_sample):,} of {len(projection):,} papers")


3D rendering sample: 13,307 of 26,103 papers


In [19]:
hover_fields = {
    "id": True,
    "title": True,
    "topics": True,
    "cluster_persistence": ":.3f",
    "umap_x": False,
    "umap_y": False,
    "umap_z": False,
}

all_topics_3d = px.scatter_3d(
    visual_sample,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="topic_code",
    hover_data=hover_fields,
    title="3D UMAP projection of all HDBSCAN topic assignments",
    opacity=0.65,
)
all_topics_3d.update_traces(marker={"size": 2})
all_topics_3d.update_layout(showlegend=False)
all_topics_3d.write_html(OUTPUT_DIR / "bertopic_umap3d_all_topics.html")

persistence_3d = px.scatter_3d(
    visual_sample,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="cluster_persistence",
    color_continuous_scale="Viridis",
    range_color=(0, max(visual_sample["cluster_persistence"].max(), 1e-6)),
    hover_data=hover_fields,
    title="3D UMAP projection coloured by HDBSCAN persistence",
    opacity=0.65,
)
persistence_3d.update_traces(marker={"size": 2})
persistence_3d.write_html(OUTPUT_DIR / "bertopic_umap3d_persistence.html")

wave_topics_3d = px.scatter_3d(
    visual_sample,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="wave_group",
    hover_data=hover_fields,
    title="3D UMAP projection: wave topics and other topics",
    opacity=0.65,
)
wave_topics_3d.update_traces(marker={"size": 2})
wave_topics_3d.write_html(OUTPUT_DIR / "bertopic_umap3d_wave_topics.html")
display(persistence_3d)


In [20]:
figure = plt.figure(figsize=(10, 7))
axis = figure.add_subplot(111, projection="3d")
points = axis.scatter(
    visual_sample["umap_x"],
    visual_sample["umap_y"],
    visual_sample["umap_z"],
    c=visual_sample["cluster_persistence"],
    cmap="viridis",
    s=4,
    alpha=0.55,
)
axis.set(
    title="3D UMAP projection coloured by HDBSCAN persistence",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    zlabel="UMAP 3",
)
figure.colorbar(points, ax=axis, pad=0.1, label="Cluster persistence")
save_figure(figure, "bertopic_umap3d_persistence_static")


## 18. Final validation and download links


In [21]:
requested_file = OUTPUT_DIR / "showcase_plus_id_topics.csv"
requested = pd.read_csv(requested_file)

assert requested.columns.tolist() == ["id", "topics"]
assert len(requested) == len(publications)
assert requested["id"].is_unique
assert requested["id"].notna().all()
assert requested["topics"].notna().all()

summary = {
    "papers": len(requested),
    "unique_ids": requested["id"].nunique(),
    "topics_in_final_model": len(final_topic_ids),
    "wave_topics": len(selected_topics),
    "wave_coverage_percent": float(coverage.loc[0, "coverage_percent"]),
    "weighted_cluster_persistence": final_persistence_metrics["weighted_cluster_persistence"],
    "low_persistence_fraction": final_persistence_metrics["low_persistence_fraction"],
    "requested_file": str(requested_file),
}
required_outputs = [
    requested_file,
    OUTPUT_DIR / "bertopic_seed_robustness_summary.csv",
    OUTPUT_DIR / "bertopic_final_cluster_persistence.csv",
    OUTPUT_DIR / "bertopic_umap3d_all_topics.html",
    OUTPUT_DIR / "bertopic_umap3d_persistence.html",
    OUTPUT_DIR / "bertopic_umap3d_wave_topics.html",
]
assert all(path.exists() and path.stat().st_size > 0 for path in required_outputs)
print(json.dumps(summary, indent=2))
print("\nAll outputs saved in:", OUTPUT_DIR)


{
  "papers": 26103,
  "unique_ids": 26103,
  "topics_in_final_model": 76,
  "wave_topics": 14,
  "wave_coverage_percent": 43.95280235988201,
  "weighted_cluster_persistence": 0.1926080186983224,
  "low_persistence_fraction": 0.5921052631578947,
  "requested_file": "/content/drive/MyDrive/UKB_BERTopic_seed_robustness/output/showcase_plus_id_topics.csv"
}

All outputs saved in: /content/drive/MyDrive/UKB_BERTopic_seed_robustness/output
